# 04_model_creation.ipynb

This notebook packages the trained XGBoost model created during the training workflow as a SageMaker-compatible model artifact.

The notebook creates a `model.tar.gz` archive, uploads it to Amazon S3, creates a SageMaker Model, and performs a temporary endpoint smoke test to verify that the model can be used for real-time inference.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries, AWS SDK clients, and SageMaker classes.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import tarfile
import time

import boto3
import joblib
import pandas as pd
import xgboost as xgb

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    MODEL_ARTIFACT_KEY,
    SAGEMAKER_MODEL_ARTIFACT_KEY,
    SAGEMAKER_MODEL_METADATA_KEY,
    SAGEMAKER_MODEL_NAME_PREFIX,
    ENDPOINT_CONFIG_NAME_PREFIX,
    TEMP_ENDPOINT_NAME_PREFIX,
    INFERENCE_INSTANCE_TYPE,
    XGBOOST_IMAGE_URI,
    TARGET_COLUMN,
    TEST_DATA_KEY,
)

## Initialize AWS Clients

Initialize the required AWS clients.

The SageMaker client is used to create the SageMaker Model, endpoint configuration, and temporary endpoint. The SageMaker Runtime client is used to invoke the endpoint.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)

## Resolve SageMaker Execution Role

Resolve the SageMaker execution role from the current Studio execution context.

This role is required when creating the SageMaker Model.

In [ ]:
identity = sts.get_caller_identity()

account_id = identity["Account"]
caller_arn = identity["Arn"]

role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]

iam = boto3.client("iam", region_name=AWS_REGION)

role_response = iam.get_role(
    RoleName=role_name,
)

execution_role_arn = role_response["Role"]["Arn"]

print(f"SageMaker execution role: {execution_role_arn}")

## Define Model Creation Configuration

Define local paths, S3 keys, SageMaker resource names, and the inference instance type.

A timestamp is added to SageMaker resource names so that the notebook can be re-run without naming conflicts.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

LOCAL_MODEL_DIR = Path("data/models")
LOCAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_MODEL_PATH = LOCAL_MODEL_DIR / "best_model.pkl"
LOCAL_XGBOOST_MODEL_PATH = LOCAL_MODEL_DIR / "xgboost-model.json"
LOCAL_MODEL_ARCHIVE_PATH = LOCAL_MODEL_DIR / "model.tar.gz"
LOCAL_FEATURE_COLUMNS_PATH = LOCAL_MODEL_DIR / "feature_columns.json"
LOCAL_MODEL_METADATA_PATH = LOCAL_MODEL_DIR / "sagemaker_model_metadata.json"

SAGEMAKER_MODEL_NAME = f"{SAGEMAKER_MODEL_NAME_PREFIX}-{timestamp}"
ENDPOINT_CONFIG_NAME = f"{ENDPOINT_CONFIG_NAME_PREFIX}-{timestamp}"
TEMP_ENDPOINT_NAME = f"{TEMP_ENDPOINT_NAME_PREFIX}-{timestamp}"

print(f"SageMaker model name: {SAGEMAKER_MODEL_NAME}")
print(f"Endpoint config name: {ENDPOINT_CONFIG_NAME}")
print(f"Temporary endpoint name: {TEMP_ENDPOINT_NAME}")
print(f"Inference instance type: {INFERENCE_INSTANCE_TYPE}")
print(f"XGBoost image URI: {XGBOOST_IMAGE_URI}")

## Download Trained Model from Amazon S3

Download the trained model artifact created in the training notebook.

The training notebook saved the best XGBoost model as a pickle file.

In [ ]:
s3.download_file(
    BUCKET_NAME,
    MODEL_ARTIFACT_KEY,
    str(LOCAL_MODEL_PATH),
)

print(f"Downloaded model from s3://{BUCKET_NAME}/{MODEL_ARTIFACT_KEY}")

## Load Trained Model

Load the trained model locally and inspect the model object.

The model is expected to be an XGBoost classifier trained with the scikit-learn API.

In [ ]:
model = joblib.load(LOCAL_MODEL_PATH)

model

## Load Test Dataset

Load the test dataset from Amazon S3.

The test dataset is used to verify local predictions, preserve the correct feature column order, and create a small endpoint invocation payload.

In [ ]:
test_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TEST_DATA_KEY,
)

test_df = pd.read_csv(test_obj["Body"])

test_df.head()

In [ ]:
X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

feature_columns = list(X_test.columns)

print(f"Test dataset shape: {test_df.shape}")
print(f"Number of features: {len(feature_columns)}")
print(f"Feature columns: {feature_columns[:10]}")

## Test Local Model Prediction

Run a small local prediction test before converting the model.

This verifies that the downloaded pickle model can still generate prediction probabilities.

In [ ]:
sample = X_test.head(3)

local_predictions = model.predict_proba(sample)

local_predictions

## Convert Trained Model to XGBoost Booster

Convert the loaded `XGBClassifier` model into its underlying XGBoost Booster representation.

The SageMaker XGBoost container can serve this model artifact directly.

In [ ]:
booster = model.get_booster()

booster.save_model(str(LOCAL_XGBOOST_MODEL_PATH))

print(f"Saved XGBoost Booster model to {LOCAL_XGBOOST_MODEL_PATH}")

## Verify Booster Prediction

Verify that the converted XGBoost Booster produces the same positive-class probabilities as the original `XGBClassifier`.

For binary classification, the Booster output should match `predict_proba(... )[:, 1]`.

In [ ]:
dmatrix_sample = xgb.DMatrix(sample)
booster_predictions = booster.predict(dmatrix_sample)

comparison_df = pd.DataFrame(
    {
        "original_model": local_predictions[:, 1],
        "converted_booster": booster_predictions,
    },
    index=sample.index,
).round(6)

comparison_df

### Verification Result

The converted XGBoost Booster produces the same prediction probabilities as the original trained model for the sample records.

This confirms that the model conversion preserved the prediction behavior.

## Create and Upload SageMaker Model Archive

Create a SageMaker-compatible `model.tar.gz` archive and upload it to Amazon S3.

The archive contains the converted XGBoost Booster model file and is used as the model artifact when creating the SageMaker Model.

In [ ]:
with tarfile.open(LOCAL_MODEL_ARCHIVE_PATH, "w:gz") as tar:
    tar.add(
        LOCAL_XGBOOST_MODEL_PATH,
        arcname="xgboost-model",
    )

print(f"Created model archive: {LOCAL_MODEL_ARCHIVE_PATH}")

s3.upload_file(
    str(LOCAL_MODEL_ARCHIVE_PATH),
    BUCKET_NAME,
    SAGEMAKER_MODEL_ARTIFACT_KEY,
)

model_data_s3_uri = f"s3://{BUCKET_NAME}/{SAGEMAKER_MODEL_ARTIFACT_KEY}"

print(f"Uploaded SageMaker model artifact to {model_data_s3_uri}")

## Create SageMaker Model

Create a SageMaker Model using the uploaded model artifact and the SageMaker XGBoost inference container.

This step creates the SageMaker Model entity, but it does not create a running endpoint yet.

In [ ]:
create_model_response = sm.create_model(
    ModelName=SAGEMAKER_MODEL_NAME,
    ExecutionRoleArn=execution_role_arn,
    PrimaryContainer={
        "Image": XGBOOST_IMAGE_URI,
        "ModelDataUrl": model_data_s3_uri,
    },
)

create_model_response

In [ ]:
model_description = sm.describe_model(
    ModelName=SAGEMAKER_MODEL_NAME,
)

print(f"SageMaker Model {model_description["ModelName"]} successfully created")

## Deploy Temporary Test Endpoint

Create an endpoint configuration and deploy a temporary endpoint to verify that the SageMaker Model can be loaded and invoked successfully.

The endpoint configuration defines the model and instance type used for hosting. The endpoint is only used as a smoke test and will be deleted after the prediction test.

In [ ]:
create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": SAGEMAKER_MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INFERENCE_INSTANCE_TYPE,
            "InitialVariantWeight": 1.0,
        }
    ],
)

create_endpoint_config_response

In [ ]:
create_endpoint_response = sm.create_endpoint(
    EndpointName=TEMP_ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)

create_endpoint_response

In [ ]:
print(f"Waiting for endpoint to be in service: {TEMP_ENDPOINT_NAME}")

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=TEMP_ENDPOINT_NAME,
)

print(f"Endpoint {TEMP_ENDPOINT_NAME} is in service.")

## Verify Temporary Endpoint Predictions

Invoke the temporary endpoint with a small sample from the test dataset and compare the endpoint predictions with the local Booster predictions.

This verifies that the deployed SageMaker Model returns the expected prediction values before the temporary endpoint is deleted.

In [ ]:
payload = sample.to_csv(
    header=False,
    index=False,
)

response = sm_runtime.invoke_endpoint(
    EndpointName=TEMP_ENDPOINT_NAME,
    ContentType="text/csv",
    Accept="text/csv",
    Body=payload,
)

prediction_response = response["Body"].read().decode("utf-8")

print(prediction_response)

In [ ]:
endpoint_predictions = [
    float(value)
    for value in prediction_response.strip().splitlines()
]

comparison_df["sagemaker_endpoint"] = endpoint_predictions

comparison_df.round(6)

### Verification Result

The temporary SageMaker endpoint returns the same prediction probabilities as the local XGBoost Booster for the sample records.

This confirms that the SageMaker Model can be loaded, hosted, and invoked successfully.

## Store SageMaker Model Metadata

Store metadata about the created SageMaker Model locally and in Amazon S3.

The next notebook can use this metadata to run SageMaker Clarify against the created model.

In [ ]:
model_metadata = {
    "sagemaker_model_name": SAGEMAKER_MODEL_NAME,
    "model_data_s3_uri": model_data_s3_uri,
    "xgboost_image_uri": XGBOOST_IMAGE_URI,
    "inference_instance_type": INFERENCE_INSTANCE_TYPE,
    "temporary_endpoint_name": TEMP_ENDPOINT_NAME,
    "endpoint_config_name": ENDPOINT_CONFIG_NAME,
    "created_at": timestamp,
    "target_column": TARGET_COLUMN,
    "feature_columns": feature_columns,
    "model_format": "xgboost-booster",
    "source_model_artifact_key": MODEL_ARTIFACT_KEY,
    "xgboost_version_notebook": xgb.__version__,
}

with open(LOCAL_MODEL_METADATA_PATH, "w") as f:
    json.dump(model_metadata, f, indent=2)

s3.upload_file(
    str(LOCAL_MODEL_METADATA_PATH),
    BUCKET_NAME,
    SAGEMAKER_MODEL_METADATA_KEY,
)

print(f"Uploaded model metadata to s3://{BUCKET_NAME}/{SAGEMAKER_MODEL_METADATA_KEY}")

model_metadata

## Clean Up Temporary Endpoint

Delete the temporary endpoint and endpoint configuration to avoid unnecessary costs.

The SageMaker Model entity is intentionally kept because it will be used in the next notebook for SageMaker Clarify.

In [ ]:
endpoint_description = sm.describe_endpoint(
    EndpointName=TEMP_ENDPOINT_NAME,
)

endpoint_config_name = endpoint_description["EndpointConfigName"]

sm.delete_endpoint(
    EndpointName=TEMP_ENDPOINT_NAME,
)

print(f"Deleting endpoint: {TEMP_ENDPOINT_NAME}")

sm.get_waiter("endpoint_deleted").wait(
    EndpointName=TEMP_ENDPOINT_NAME,
)

print(f"Deleted endpoint: {TEMP_ENDPOINT_NAME}")

In [ ]:
try:
    sm.describe_endpoint(
        EndpointName=TEMP_ENDPOINT_NAME,
    )
except sm.exceptions.ClientError as error:
    print("Temporary endpoint no longer exists.")

In [ ]:
model_description = sm.describe_model(
    ModelName=SAGEMAKER_MODEL_NAME,
)

print(f"SageMaker Model {model_description["ModelName"]} kept for Clarify")

## Result

The locally trained XGBoost model was converted into a SageMaker-compatible XGBoost model artifact and uploaded to Amazon S3.

A SageMaker Model was created using the SageMaker XGBoost inference container. A temporary endpoint was deployed and invoked successfully as a smoke test. The temporary endpoint and endpoint configuration were deleted after the test.

The SageMaker Model entity is kept and can be used in the next notebook for SageMaker Clarify.